# Automated Malaria Cell Detection — Phase 2, Milestone 2
## Full 10-fold reproduction of Marques *et al.* (2022), following the paper's own code

**Team:** J Amathziah (230139): Kaggle runs, repo · C Murali Madhav (230115): code (notebook, scripts) · Ravi Yadav (230131): slides · Pugazhendhi J (230066): report
**Base paper:** G. Marques, A. Ferreras, I. de la Torre-Díez, "An ensemble-based approach for automated medical diagnosis of malaria using EfficientNet," *Multimedia Tools and Applications*, 81, 28061–28078, 2022.
**The paper's code:** Springer supplementary file [MOESM1](https://static-content.springer.com/esm/art%3A10.1007%2Fs11042-022-12624-6/MediaObjects/11042_2022_12624_MOESM1_ESM.pdf), the authors' executed Jupyter notebook (`efn0_malaria_binary`, 30 Jan 2021).

Our first reproduction (`Phase2_Baseline_Reproduction_Marques2022.ipynb`, experiment A0, 3 of 10 folds) followed the paper's *text*. The supplementary code shows a recipe that differs in several places. This notebook follows the **code** and trains **all 10 folds**:

| Setting | Paper's code (this notebook) | First reproduction (A0, 3 folds) |
|---|---|---|
| Pretrained weights | **noisy-student**, `efficientnet` 1.1.1 (qubvel) | ImageNet, `tf.keras.applications` |
| Head | GAP → Dense 128 → Dropout 0.3 → Dense 64 → Dropout 0.3 → Dense 32 → Dropout 0.3 → Dense 2 (softmax) | GAP → Dropout 0.2 → Dense 1 (sigmoid) |
| Loss | categorical cross-entropy, **label smoothing 0.1** | binary cross-entropy |
| Optimiser, schedule | Adam 1e-4; ReduceLROnPlateau(val_loss, **factor 0.5**, patience 6, min 1e-6); **33 epochs**, keep the best-val_loss weights | Adam 1e-4; factor 0.1; ≤ 15 epochs with early stopping |
| Batch size | **16** (1,240 steps per epoch) | 64 per GPU (128) |
| Hold-out ("validation") set | **20 %** = 5,512 images | 10 % = 2,756 images |
| Folds | StratifiedKFold(10, shuffle, random_state=50) on the other 80 % | StratifiedKFold(10, seed 42) |
| Augmentation | albumentations 0.5.2 `Compose`: rotate 90/flip/transpose, noise, blur, ShiftScaleRotate, optical/grid/piecewise-affine distortion, CLAHE/sharpen/emboss/contrast/brightness | flips + 90° rotations |
| Input | cv2 nearest-neighbour resize to 224, pixels / 255 | bilinear resize, Keras ImageNet normalisation |
| Evaluation images | **augmented too** (the same generator feeds test and validation) | un-augmented |

The results go to `results/A0_10fold_paper/`; `results/A0_baseline/` is not touched.

### Targets: what the paper's code printed
| Metric | Paper |
|---|---|
| Mean accuracy of the 10 fold models on their CV test part (Table 5) | 97.56 % |
| Mean accuracy of the 10 fold models on the 5,512-image hold-out set (Table 6) | 97.69 % (Table 6 prints 97.70) |
| **Ensemble accuracy** (hold-out) | **98.29 %** |
| Ensemble recall, parasitized | 97.74 % |
| Ensemble precision, parasitized | 98.82 % |
| Ensemble F1, parasitized | 98.28 % |
| Ensemble ROC-AUC | 99.76 % |

The paper's text swaps the last two ("recall 98.82 %, precision 97.74 %"). Its own printed report (2,744 parasitized cells, 62 missed; 2,768 uninfected, 32 flagged) gives recall 97.74 % and precision 98.82 %.

### How to run
* **RunPod, one pod per fold (what we did):** `python src/runpod_10fold.py up --folds 0-9`, then `python src/runpod_10fold.py watch`. Each pod (RTX 4090) runs this notebook with `FOLDS_TO_TRAIN=[k]`; `watch` copies every finished fold into `results/A0_10fold_paper/folds/`, moves the weights to `models/A0_10fold_paper/` and terminates the pod. Then run this notebook with `FOLDS_TO_TRAIN = []` to build the tables.
* **One GPU machine:** `FOLDS_TO_TRAIN = list(range(10))`, Run All.
* **Environment:** TensorFlow 2.15 (the last Keras 2), `efficientnet==1.1.1`, `albumentations==0.5.2`, `imgaug==0.4.0`, `numpy==1.23.5`; the exact pip line is in `src/runpod_10fold.py`. The recipe lives in `src/marques2022_exact.py` and is checked against the paper by `tests/test_marques2022_exact.py`.

## 1 · Parameters

In [ ]:
FOLDS_TO_TRAIN = []          # [k] on a pod, list(range(10)) on one machine, [] = only build the tables
AGGREGATE = True             # build the paper's tables when all 10 folds are in OUT_DIR
DATA_DIR = "../outputs/data"                           # NIH cell_images (downloaded if missing; git-ignored)
CACHE_PATH = "../outputs/cell_images_224_uint8.npy"    # decoded images, 4.1 GB (git-ignored)
OUT_DIR = "../results/A0_10fold_paper/folds"           # one sub-folder per fold
RESULTS_DIR = "../results/A0_10fold_paper"
N_WORKERS = None             # augmentation processes; None = CPUs the container allows - 2
XLA = True                   # XLA-compile the training step (speed only)
SMOKE = False                # quick pipeline test: 5 % of the images, 2 epochs of 20 steps

In [ ]:
%matplotlib inline
import json, os, sys, time
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")
os.environ.setdefault("TF_GPU_THREAD_MODE", "gpu_private")  # dedicated host threads launch the GPU kernels
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)

SRC = next(p for p in ["../src", "src", "/workspace/repo/src"] if os.path.exists(os.path.join(p, "marques2022_exact.py")))
sys.path.insert(0, os.path.abspath(SRC))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve
import marques2022_exact as mx

TRAINING = len(FOLDS_TO_TRAIN) > 0
print("Recipe (from the paper's code):", json.dumps(mx.PAPER))
print("Folds to train here:", FOLDS_TO_TRAIN or "none (tables only)")

## 2 · Data and the paper's split
NIH malaria cell images, 27,558 cells. As in the paper's code (section 1.3):
1. all image files of both classes (non-images such as `Thumbs.db` are dropped);
2. `train_test_split(test_size=0.2)` → 22,046 images for the 10-fold CV and an untouched hold-out set of 5,512 images, which the paper calls the "validation" set;
3. `StratifiedKFold(10, shuffle=True, random_state=50)` on the 22,046 images → 19,841 or 19,842 training images and 2,205 or 2,204 test images per fold.

The paper's split had no `random_state` and was not stratified (its hold-out set had 2,744 parasitized and 2,768 uninfected cells). We keep it unstratified and sort the file list and fix `random_state=1234` (the paper's `SEED`), so that the 10 pods build the same split; a hash of the split is stored with every fold and checked before the tables are built.

In [ ]:
if TRAINING:
    root = mx.find_or_download(DATA_DIR)
    df = mx.index_dataset(root)
    if SMOKE:  # like the paper's QUICK_AND_DIRTY switch
        df = mx.build_index(df.sample(frac=0.05, random_state=mx.PAPER["SEED"])[["file", "label"]].values, root)
        CACHE_PATH = CACHE_PATH.replace(".npy", "_smoke.npy")
    learn, val = mx.paper_split(df)
    folds = mx.paper_folds(learn)
    print(f"Images: {len(df):,}", df["label"].value_counts().to_dict())
    display(pd.DataFrame({
        "paper": ["22,046", "5,512", "2,744 / 2,768", "19,841 or 19,842", "2,205 or 2,204"],
        "ours": [f"{len(learn):,}", f"{len(val):,}",
                 " / ".join(f"{(val.label == t).sum():,}" for t in mx.TIPOS),
                 " or ".join(sorted({f"{len(tr):,}" for tr, _ in folds})),
                 " or ".join(sorted({f"{len(te):,}" for _, te in folds}, reverse=True))]},
        index=["CV (learning) set", "hold-out set", "hold-out parasitized / uninfected",
               "training images per fold", "test images per fold"]))
    print("Split fingerprint:", mx.split_fingerprint(learn, val, folds))

## 3 · The paper's augmentation
Section 2.3 of the paper's code, verbatim (`tests/` checks it against a copy of the original). The images are read and resized the way `ImageDataAugmentor` does it (cv2, nearest neighbour, 224 × 224), augmented as uint8, then divided by 255.

The paper's `ImageDataAugmentor` object was used for the training, test **and** validation generators, so every prediction the paper reports was made on a randomly augmented image. We do the same for the headline numbers and also report every metric on clean (un-augmented) images.

In [ ]:
if TRAINING:
    aug = mx.build_paper_augmentation()
    print(aug)
    import random, imgaug
    row = df.sample(2, random_state=3)
    fig, axes = plt.subplots(2, 7, figsize=(15, 4.6))
    for r, (path, label) in enumerate(zip(row["path"], row["label"])):
        img = mx.load_resized(path)
        axes[r, 0].imshow(img); axes[r, 0].set_title(f"{label}\noriginal", fontsize=9)
        for c in range(1, 7):
            random.seed(c); np.random.seed(c); imgaug.random.seed(c)
            axes[r, c].imshow(aug(image=img)["image"]); axes[r, c].set_title(f"augmented {c}", fontsize=9)
    for a in axes.ravel(): a.axis("off")
    plt.tight_layout(); plt.show()

## 4 · Model: EfficientNet-B0 (noisy-student) + the paper's head
`create_model()` from the paper's code: EfficientNet-B0 without its top (all layers trainable) → global average pooling → Dense 128 → Dropout 0.3 → Dense 64 → Dropout 0.3 → Dense 32 → Dropout 0.3 → Dense 2 (softmax). Compiled with Adam (1e-4) and categorical cross-entropy with label smoothing 0.1. The paper's Table 2 lists 4,223,934 parameters, 4,181,918 of them trainable.

In [ ]:
if TRAINING:
    import tensorflow as tf
    for g in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(g, True)
    print("TensorFlow", tf.__version__, "| GPUs:", [tf.config.experimental.get_device_details(g).get("device_name")
                                                 for g in tf.config.list_physical_devices("GPU")])
    m = mx.build_model(weights=None)
    trainable = int(sum(np.prod(w.shape) for w in m.trainable_weights))
    display(pd.DataFrame({"paper (Table 2)": [4_223_934, 4_181_918, 42_016],
                          "ours": [m.count_params(), trainable, m.count_params() - trainable]},
                         index=["total parameters", "trainable", "non-trainable"]))
    del m; tf.keras.backend.clear_session()

## 5 · Train the folds
Each fold is the loop body of the paper's section 2.4: 33 epochs × 1,240 steps of 16 images, validation on the fold's test part after every epoch, ReduceLROnPlateau, the best-val_loss checkpoint reloaded at the end, then predictions on the test part and on the 5,512 hold-out images.

**Speed without changing the maths.** The paper needed 4 h 46 min per fold on average (4:35 to 4:58) on a GTX 1050. Here:
* every image is decoded and resized **once** into a uint8 array that all processes share (the paper re-read every PNG every epoch);
* the albumentations pipeline runs in a **pool of worker processes**, each batch seeded by (fold, epoch, batch), so the batches are reproducible and identical for any number of workers;
* batches cross PCIe as **uint8** (4× less than float32) and are divided by 255 on the GPU, inside the model;
* the training step is **XLA-compiled** (fused kernels; at batch 16 kernel-launch overhead is most of the step time). Evaluation stays uncompiled, so its smaller last batch does not trigger a recompile;
* float32 maths as in the paper (no mixed precision); TF32 tensor cores as TensorFlow sets them on Ampere and newer GPUs;
* on RunPod, the 10 folds run **at the same time, one RTX 4090 per fold**.

In [ ]:
def print_like_the_paper(res):
    """The paper's per-fold printout: accuracy + classification report, test part then hold-out."""
    for name, title in [("test_aug", "Predicting... (CV test part)"), ("val_aug", "Validating... (hold-out set)")]:
        npz = np.load(os.path.join(OUT_DIR, f"fold_{res['fold']}", f"fold_{res['fold']}_predictions.npz"))
        y_true = npz["test_true" if name.startswith("test") else "val_true"]
        pred = res["probs"][name].argmax(1)
        print(title, f"\nAccuracy {np.mean(pred == y_true):.6f}")
        print(classification_report(y_true, pred, target_names=mx.TIPOS, digits=6))

if TRAINING:
    workers = N_WORKERS or max(1, mx.available_cpus() - 2)
    if not os.path.exists(CACHE_PATH):
        t0 = time.time()
        mx.build_image_cache(list(df["path"]), CACHE_PATH, workers=mx.available_cpus())
        print(f"Image cache built in {time.time() - t0:.0f} s")
    smoke = dict(epochs=2, max_steps_per_epoch=20) if SMOKE else {}
    with mx.AugmentPool(workers, CACHE_PATH) as pool:
        for k in FOLDS_TO_TRAIN:
            print("*" * 50, f"\n** Model: EfficientNetB0    fold: {k}   ({workers} augmentation workers)")
            print(f"Found {len(folds[k][0])} / {len(folds[k][1])} / {len(val)} images (train / test / validation)")
            t0 = time.time()
            res = mx.train_fold(k, learn=learn, val=val, folds=folds, pool=pool, out_dir=OUT_DIR, xla=XLA, **smoke)
            print_like_the_paper(res)
            print(f"Best epoch {res['best_epoch']} of {res['epochs']} · training {res['train_seconds'] / 60:.1f} min "
                  f"(median epoch {np.median(res['epoch_seconds']):.0f} s) · End of Fold {k} - Elapsed Time: "
                  f"{time.strftime('%H:%M:%S', time.gmtime(time.time() - t0))}")

## 6 · Results: the paper's tables, theirs vs. ours
Built from the saved outputs of all 10 folds. "Paper protocol" = predictions on augmented images, exactly as the paper's code made them; "clean" = the same 10 models on the original images.

In [ ]:
done = [k for k in range(10) if os.path.exists(os.path.join(OUT_DIR, f"fold_{k}", f"fold_{k}.json"))]
READY = AGGREGATE and len(done) == 10 and not SMOKE
print(f"Folds available: {done}" + ("" if READY else "  → tables need all 10 folds"))
if READY:
    agg = mx.aggregate(OUT_DIR)
    summary = mx.write_results(agg, RESULTS_DIR)
    pf = agg["per_fold"]["paper"]
    print("All folds share split", agg["split_fingerprint"], "| images:", agg["n_images"],
          "| GPUs:", sorted(set(pf["gpu"])), "| training minutes per fold:", pf["minutes"].tolist())

### Tables 3 and 4: each class on the fold's CV test part (paper protocol)

In [ ]:
if READY:
    t34 = pd.DataFrame({
        "malaria precision": pf["cv_precision"], "malaria recall": pf["cv_recall"], "malaria F1": pf["cv_f1"],
        "normal precision": pf["cv_uninfected_precision"], "normal recall": pf["cv_specificity"],
        "normal F1": pf["cv_uninfected_f1"]}, index=pf["fold"]) / 100
    t34.loc["Average"] = t34.mean()
    t34.loc["Paper average"] = [0.980687, 0.970457, 0.975535, 0.970731, 0.980837, 0.976648]
    display(t34.round(6))

### Table 5 (CV test part) and Table 6 (each fold model on the hold-out set), weighted averages

In [ ]:
if READY:
    t56 = pd.DataFrame({
        "T5 accuracy": pf["cv_accuracy"] / 100, "T5 accuracy (paper)": mx.PAPER_RESULTS["cv_accuracy"],
        "T5 precision": pf["cv_weighted_precision"] / 100, "T5 F1": pf["cv_weighted_f1"] / 100,
        "T6 accuracy": pf["holdout_accuracy"] / 100, "T6 accuracy (paper)": mx.PAPER_RESULTS["holdout_accuracy"],
        "T6 precision": pf["holdout_weighted_precision"] / 100, "T6 F1": pf["holdout_weighted_f1"] / 100,
        "T6 ROC": pf["holdout_auc"] / 100, "T6 ROC (paper)": mx.PAPER_RESULTS["holdout_auc"],
        "best epoch": pf["best_epoch"]}, index=pf["fold"])
    t56.loc["Average"] = t56.mean()
    display(t56.round(6))

### The 10-model ensemble on the 5,512-image hold-out set

In [ ]:
if READY:
    for proto in ["paper", "clean"]:
        p = agg["ens_probs"][proto]
        pred = p.argmax(1)
        print(f"ENSEMBLE ({proto} protocol)\nAccuracy {np.mean(pred == agg['val_true']):.6f}")
        print(classification_report(agg["val_true"], pred, target_names=mx.TIPOS, digits=6))
        e = summary["ensemble" if proto == "paper" else "ensemble_clean"]
        print(f"Real: Parasitized -> Predicted: Uninfected   NUM. ERRORS: {e['FN']}   (paper: 62)")
        print(f"Real: Uninfected -> Predicted: Parasitized   NUM. ERRORS: {e['FP']}   (paper: 32)\n")

### Ours vs. the paper

In [ ]:
if READY:
    display(agg["comparison"])

In [ ]:
if READY:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    for r in agg["records"]:
        h = r["history"]
        axes[0].plot(range(1, len(h["val_accuracy"]) + 1), h["val_accuracy"], alpha=.8, label=f"fold {r['fold']}")
        axes[1].plot(range(1, len(h["val_loss"]) + 1), h["val_loss"], alpha=.8)
    axes[0].set_title("Accuracy on the fold's test part, per epoch"); axes[0].set_xlabel("epoch"); axes[0].legend(fontsize=7, ncol=2)
    axes[1].set_title("val_loss (drives the LR schedule and the checkpoint)"); axes[1].set_xlabel("epoch")
    cm = confusion_matrix(agg["val_true"], agg["ens_probs"]["paper"].argmax(1), labels=[0, 1])
    axes[2].imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        axes[2].text(j, i, f"{v:,}", ha="center", va="center", fontsize=14, color="white" if v > cm.max() / 2 else "black")
    axes[2].set_xticks([0, 1], mx.TIPOS); axes[2].set_yticks([0, 1], mx.TIPOS)
    axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("True"); axes[2].set_title("Ensemble confusion matrix (hold-out, paper protocol)")
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "A0_10fold_curves_cm.png"), dpi=140); plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
    for r in agg["records"]:
        fpr, tpr, _ = roc_curve(agg["val_true"] == 0, r["npz"]["val_aug"][:, 0]); axes[0].plot(fpr, tpr, alpha=.4)
    fpr, tpr, _ = roc_curve(agg["val_true"] == 0, agg["ens_probs"]["paper"][:, 0])
    axes[0].plot(fpr, tpr, lw=2.5, color="black", label=f"ensemble (AUC {100 * agg['ensemble']['paper']['auc']:.2f} %)")
    axes[0].set_xlim(0, .3); axes[0].set_ylim(.7, 1.01); axes[0].legend(); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
    axes[0].set_title("ROC on the hold-out set (thin: single fold models)")
    c = agg["comparison"]; xs = np.arange(len(c)); w = .38
    axes[1].bar(xs - w / 2, c["Paper"], w, label="Marques et al. (2022), from their code", color="#9FC0C9")
    axes[1].bar(xs + w / 2, c["Ours (paper)"], w, label="Ours, 10 folds, paper protocol", color="#137C8E")
    for x, v in zip(xs + w / 2, c["Ours (paper)"]): axes[1].text(x, v + .05, f"{v:.2f}", ha="center", fontsize=7)
    axes[1].set_xticks(xs, [m.replace(" (", "\n(") for m in c["Metric"]], rotation=90, fontsize=7)
    axes[1].set_ylim(min(96, c[["Paper", "Ours (paper)"]].min().min() - .5), 100.2)
    axes[1].legend(fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False)
    axes[1].set_title("Paper vs. ours (axis starts above 0)", pad=24)
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "A0_10fold_roc_vs_paper.png"), dpi=140); plt.show()

## 7 · Fidelity notes
**Identical to the paper's code:** the data and the 80/20 + stratified 10-fold protocol, the model and its weights (noisy-student, qubvel `efficientnet` 1.1.1, same Table 2 parameter counts), the loss with label smoothing, Adam 1e-4, ReduceLROnPlateau (factor 0.5, patience 6, min 1e-6), 33 epochs of 1,240 steps with batch 16, best-val_loss checkpoint, the albumentations 0.5.2 pipeline (with imgaug 0.4.0 for the `IAA*` transforms), the cv2 nearest-neighbour resize and /255 scaling, augmented evaluation, the averaging ensemble, and the sklearn reports.

**Different, and why it should not matter:**
1. TensorFlow 2.15 instead of 2.3 (modern GPUs need it; same Keras 2 API, same Adam implementation as `tf.keras.optimizers.legacy.Adam`), RTX 4090 instead of a GTX 1050, XLA-compiled training step, TF32 convolutions.
2. The hold-out split is seeded and the file list sorted (the paper's was unseeded, so its exact split cannot be recovered); augmentation and shuffling are seeded per batch instead of drawing from one global random stream.
3. Each epoch uses exactly 1,240 full batches (the paper's stated count; 1 or 2 images are left out of each epoch, differently every epoch).

**Two things in the paper itself:** its text swaps the ensemble's precision and recall, and Table 6's "accuracy" column is (precision + recall) / 2 of the printed reports (97.70 instead of 97.69 on average). We compare against the printed reports.

## 8 · Conclusion
* **The reproduction succeeds.** With the paper's own recipe on all 10 folds, the fold models score **97.55 %** on their CV test parts (paper: 97.56 %), and the 10-model ensemble reaches **97.62 %** on the 5,512-image hold-out set (paper: 98.29 %) with a ROC-AUC of **99.73 %** (paper: 99.76 %). Every headline metric is within 1 percentage point of the paper (`python src/check_results.py results/A0_10fold_paper`).
* **Where we are lower:** single models on the hold-out set are 0.32 pp below the paper (97.37 vs. 97.69 %), and our ensemble gains +0.25 pp over its average member against the paper's +0.60 pp, which leaves the ensemble 0.67 pp short. That gap is about three standard errors of a 5,512-image test set, so not all of it is sampling noise; the unrecoverable (unseeded) split of the paper, TensorFlow 2.15 instead of 2.3 and different GPUs are the remaining differences.
* **Malaria recall** (sensitivity) is 97.07 % against the paper's 97.74 % (the paper's text says 98.82 %, which is its precision). The first reproduction's "recall 2 pp lower" compared against the swapped number; against the right one this run is 0.67 pp lower.
* **Augmented vs. clean evaluation:** scoring the same models on un-augmented images changes little (ensemble 97.64 %, mean CV fold 97.78 %), so the paper's augmented evaluation does not explain its numbers.
* **Compute:** the 10 folds trained in parallel, one RTX 4090 each, in about 11 minutes per fold; the paper needed 4 h 46 min per fold on average (1 day 23 h 39 min in total) on a GTX 1050.